# SVG-aware editorial work: look, locate, patch, render

Block 2, minutes **10–40**, alongside `02_live_editing.ipynb`. Start from
the existing linked Schumann project. This notebook makes a small local packet
for comparing source/engraving images with SVG source and musical structure.
It makes **no model-provider calls**. Your own model/extension receives only
what you explicitly attach or give it access to in the next step.

The research companion `svg-editorial-workflow.md` separates raw SVG reading,
rendered-image inspection and SVG with explicit MEI attributes. The printed
witness supplies the reading; the current SVG may faithfully depict a mistake.
Packet files containing the scan stay in the ignored working project.

In [ ]:
%pip install -q "camat[validation,notebooks]==0.2.8" verovio==6.3.0
# For PNG rendering: use system rsvg-convert, or install cairosvg with system Cairo available.

In [1]:
# Helper functions for this notebook. Run this cell; you do not need to read it.
# They follow the same rules as the demos on the course page.

"""Export local score/scan context for the SVG editorial exercise.

No provider requests. Source pixels stay in the student's existing work project.
"""

import copy
import hashlib
import io
import json
from pathlib import Path
import shutil
import subprocess
import tempfile
import xml.etree.ElementTree as ET

from PIL import Image, ImageChops
import verovio

SVG_MEI_NS = "http://www.music-encoding.org/ns/mei"
SVG_NS = "http://www.w3.org/2000/svg"
SVG_XML_ID = "{http://www.w3.org/XML/1998/namespace}id"
SVG_M = "{" + SVG_MEI_NS + "}"
SVG_EXTRA_ATTRIBUTES = [
    "measure@n", "staff@n", "layer@n", "note@pname", "note@oct", "note@dur",
    "note@dots", "note@accid", "note@accid.ges", "note@stem.dir", "chord@dur",
    "chord@dots", "accid@accid", "slur@startid", "slur@endid", "slur@staff",
    "slur@curvedir", "dynam@startid", "dynam@tstamp", "dynam@staff", "dynam@place",
]


def svg_file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def svg_raster(svg_text, width=1900):
    """Use librsvg if installed; CairoSVG is the optional Python fallback."""
    executable = shutil.which("rsvg-convert")
    if executable:
        result = subprocess.run([executable, "-w", str(width), "-b", "white"],
                                input=svg_text.encode(), capture_output=True, check=True, timeout=60)
        return result.stdout, "rsvg-convert"
    try:
        import cairosvg
    except ImportError as error:
        raise ImportError("Rendering PNG needs rsvg-convert or CairoSVG (python -m pip install cairosvg); CairoSVG also needs system Cairo.") from error
    return cairosvg.svg2png(bytestring=svg_text.encode(), output_width=width,
                           background_color="white"), "CairoSVG " + cairosvg.__version__


def svg_render_selection(mei, first_position, last_position, enriched=False):
    toolkit = verovio.toolkit()
    options = {"header": "none", "footer": "none", "breaks": "none", "pageWidth": 2400,
               "pageHeight": 10000, "adjustPageHeight": True, "scale": 60,
               "svgViewBox": True, "xmlIdSeed": 1}
    if enriched:
        options["svgAdditionalAttribute"] = SVG_EXTRA_ATTRIBUTES
    toolkit.setOptions(options)
    if not toolkit.loadData(Path(mei).read_text()):
        raise ValueError("Verovio could not load the MEI")
    if not toolkit.select({"measureRange": f"{first_position}-{last_position}"}):
        raise ValueError("Verovio could not select the passage")
    toolkit.redoLayout()
    if toolkit.getPageCount() != 1:
        raise ValueError("Use a smaller passage for this one-system packet")
    return toolkit.renderToSVG(1), {"verovio": toolkit.getVersion(), "options": options}


def svg_editorial_prompt(condition):
    extras = {
        "A": "No SVG source is supplied in this condition.",
        "B": "Also read score.svg as SVG SOURCE TEXT. Its element IDs/classes address the current MEI; retain glyph definitions when inspecting it.",
        "C": "Also read score-enriched.svg as SVG SOURCE TEXT. Its data-* attributes copy current encoded values, which may be wrong or inherited from a containing chord/key; they are not evidence from the print.",
    }
    return '''Compare source-system.png (IMSLP 51556, printed p.8, first system)
with engraving.png (our current Schumann Op.68 No.10 encoding). Read context.xml,
a READ-ONLY fragment with scoreDef and MEI labels 1–5; it is not a complete MEI.
The pickup is label 1; labels 2 and 3 are the first two full bars.

Focus ONLY on the slur phrases beginning in that pickup and those two full bars.
Identify up to two missing or differently attached printed slurs. For each:
state what you see in the source, the staff, start/end notes and existing MEI
IDs, the corresponding current-engraving discrepancy, and any uncertainty.
If you cannot inspect an attachment or distinguish an endpoint, say so.
Do not infer the source reading from a familiar edition or the current metadata.
Use only the supplied task files; do not consult answer keys, other editions or the web.

Then propose the smallest MEI 5.1 insertion/correction and an explanation for
editorial review. Preserve all existing pitches, rhythms, voices, IDs, metadata,
facsimile links and source breaks. Do not change SVG paths as the edition.
Keep source reading, encoding decision and engraving/renderer problem separate.
Do not invent validation results or human approval.

Return: observations with evidence -> exact targets/uncertainties -> minimal
candidate patch -> what the human should compare after rendering.
''' + "\n" + extras[condition] + "\n"


def export_svg_editorial_packet(mei_path, output, first_label="1", last_label="5"):
    mei = Path(mei_path).resolve()
    output = Path(output).resolve()
    input_hash = svg_file_hash(mei)
    root = ET.parse(mei).getroot()
    measures = root.findall("m:music/m:body//m:measure", {"m": SVG_MEI_NS})
    labels = [measure.get("n") for measure in measures]
    if len(set(labels)) != len(labels):
        raise ValueError("This single-movement exercise needs unique MEI measure labels")
    first, last = labels.index(first_label), labels.index(last_label)
    if first > last:
        raise ValueError("Passage end precedes its start")
    selected = measures[first:last + 1]
    if not selected or any(not m.get(SVG_XML_ID) for m in selected):
        raise ValueError("Selected measures need existing IDs")
    by_id = {e.get(SVG_XML_ID): e for e in root.iter() if e.get(SVG_XML_ID)}
    parents = {child: parent for parent in root.iter() for child in parent}
    zones = [by_id[m.get("facs", "").removeprefix("#")] for m in selected]
    surfaces = {parents[zone] for zone in zones}
    if len(surfaces) != 1:
        raise ValueError("Use one printed system/page for this exercise")
    surface = next(iter(surfaces))
    graphic = surface.find(SVG_M + "graphic")
    image_path = (mei.parent / graphic.get("target")).resolve()
    image_hash = svg_file_hash(image_path)
    identity = {"input_mei_sha256": input_hash, "source_image_sha256": image_hash,
                "labels": labels[first:last + 1], "packet_format": 1}
    if output.exists():
        manifest = json.loads((output / "manifest.json").read_text())
        if manifest["identity"] != identity:
            raise FileExistsError("Existing packet describes a different input; choose a new attempt folder")
        for filename, expected in manifest["files_sha256"].items():
            if svg_file_hash(output / filename) != expected:
                raise FileExistsError(f"Preserving modified packet file: {filename}")
        return manifest

    output.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix=".svg-packet-", dir=output.parent) as temporary:
        stage = Path(temporary)
        ordinary, normal_info = svg_render_selection(mei, first + 1, last + 1)
        enriched, enriched_info = svg_render_selection(mei, first + 1, last + 1, True)
        (stage / "score.svg").write_text(ordinary)
        (stage / "score-enriched.svg").write_text(enriched)
        normal_root = ET.fromstring(ordinary)
        enriched_root = ET.fromstring(enriched)
        expected_ids = [m.get(SVG_XML_ID) for m in selected]
        def measure_ids(svg):
            return [e.get("id") for e in svg.iter()
                    if "measure" in e.get("class", "").split()]
        if measure_ids(normal_root) != expected_ids or measure_ids(enriched_root) != expected_ids:
            raise AssertionError("Rendered selection does not match MEI measures, including the pickup")
        rendered_ids = {e.get("id") for e in normal_root.iter() if e.get("id")}
        mapping = []
        for measure in selected:
            for event in measure.iter():
                if event.tag not in {SVG_M + tag for tag in ("note", "chord", "rest", "slur", "dynam", "tempo")}:
                    continue
                event_id = event.get(SVG_XML_ID)
                mapping.append({"mei_id": event_id, "measure": measure.get("n"),
                                "element": event.tag.split("}")[-1],
                                "encoded_attributes": dict(event.attrib),
                                "present_in_default_svg": event_id in rendered_ids})
                if event_id and event_id not in rendered_ids:
                    raise AssertionError(f"Expected event missing from rendered SVG: {event_id}")
        (stage / "element-map.json").write_text(json.dumps(mapping, indent=2) + "\n")
        context = ET.Element("score-context", {"purpose": "read-only MEI fragment; not a valid standalone MEI"})
        score_def = root.find("m:music/m:body//m:scoreDef", {"m": SVG_MEI_NS})
        if score_def is None:
            raise ValueError("Expected initial scoreDef for this Schumann passage")
        context.append(copy.deepcopy(score_def))
        for measure in selected:
            context.append(copy.deepcopy(measure))
        (stage / "context.xml").write_bytes(ET.tostring(context, encoding="utf-8", xml_declaration=True))
        png, renderer = svg_raster(ordinary)
        enriched_png, _ = svg_raster(enriched)
        before_pixels = Image.open(io.BytesIO(png)).convert("RGB")
        after_pixels = Image.open(io.BytesIO(enriched_png)).convert("RGB")
        same_pixels = before_pixels.size == after_pixels.size and ImageChops.difference(before_pixels, after_pixels).getbbox() is None
        if not same_pixels:
            raise AssertionError("Added data attributes unexpectedly changed the rendered appearance")
        (stage / "engraving.png").write_bytes(png)
        with Image.open(image_path) as image:
            crop = (max(0, min(int(z.get("ulx")) for z in zones) - 125),
                    max(0, min(int(z.get("uly")) for z in zones) - 64),
                    min(image.width, max(int(z.get("lrx")) for z in zones) + 125),
                    min(image.height, max(int(z.get("lry")) for z in zones) + 64))
            image.crop(crop).save(stage / "source-system.png")
        for condition in "ABC":
            (stage / f"prompt-{condition}.md").write_text(svg_editorial_prompt(condition))
        record = {"status": "not_run", "condition": "", "model_id": "", "extension_and_version": "",
                  "actual_input_transport": "unknown: record source text / raster preview / both",
                  "input_mei_sha256": input_hash, "manifest": "manifest.json",
                  "prompt_file": "", "response_file": "", "patch_file": "", "elapsed_seconds": None,
                  "usage_or_quota_observation": "", "found_phrase_divisions": "not_reviewed",
                  "correct_staff_and_endpoint_ids": "not_reviewed", "unexpected_changes": [],
                  "schema_reference_resource_checks": "not_run", "render_check": "not_run",
                  "human_source_review": "not_reviewed", "reviewer": "", "decision_and_corrections": ""}
        (stage / "record-template.json").write_text(json.dumps(record, indent=2) + "\n")
        manifest = {"prepared_on": "2026-10-26", "identity": identity,
                    "input_mei": str(mei), "source_image": str(image_path),
                    "source": "IMSLP 51556, original/excerpt/printed pages 8/1/8",
                    "source_crop_pixels": list(crop), "source_crop_basis": "union of linked measure zones plus context margin; review before use",
                    "measures_in_svg": expected_ids, "events_mapped": len(mapping),
                    "ordinary": normal_info, "enriched": enriched_info,
                    "png_renderer": renderer, "normal_and_enriched_pixels_identical": same_pixels,
                    "glyph_definitions_retained": normal_root.find("{" + SVG_NS + "}defs") is not None,
                    "source_human_review": "not_asserted", "provider_calls": 0,
                    "files_sha256": {p.name: svg_file_hash(p) for p in sorted(stage.iterdir())}}
        (stage / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
        stage.rename(output)
    if svg_file_hash(mei) != input_hash or svg_file_hash(image_path) != image_hash:
        raise AssertionError("Input changed during packet generation")
    return manifest

In [2]:
from IPython.display import display, Image as NotebookImage

PROJECT = (Path.cwd() / "schumann_edition_work").resolve()
INPUT_MEI = PROJECT / "edition/mei/work.mei"
PACKET = PROJECT / "edition/experiments/svg-01"
if not INPUT_MEI.is_file():
    raise FileNotFoundError(f"Run linking first, or set PROJECT to its existing project: {INPUT_MEI}")
MANIFEST = export_svg_editorial_packet(INPUT_MEI, PACKET)
print("Packet:", PACKET)
print("Input SHA-256:", MANIFEST["identity"]["input_mei_sha256"])
print("Measures, including pickup:", MANIFEST["measures_in_svg"])
print("Mapped events:", MANIFEST["events_mapped"])
print("Extra data attributes leave rendered pixels identical:", MANIFEST["normal_and_enriched_pixels_identical"])
print("SVG glyph definitions retained:", MANIFEST["glyph_definitions_retained"])

Packet: /tmp/camat-schumann-svg/notebook/schumann_edition_work/edition/experiments/svg-01
Input SHA-256: ad594ea784895a4cb1ded45e9ca311049c346be26711de163b8de6bf7d200885
Measures, including pickup: ['d1e9', 'd1e74', 'd1e352', 'd1e648', 'd1e968']
Mapped events: 100
Extra data attributes leave rendered pixels identical: True
SVG glyph definitions retained: True


## 10–15 — The same note as pixels, SVG and MEI

Inspect the source and current engraving. Find the pickup, then inspect the
SVG group's attributes below. `d1e60` is the same note address in the MEI and
SVG. Extra `data-*` values copy the current encoding, not a fresh reading of
the scan; key/accidental context and containing chord durations still matter.
The SVG paths and glyph definitions remain in the packet files.

In [3]:
display(NotebookImage(filename=str(PACKET / "source-system.png"), width=1500))
display(NotebookImage(filename=str(PACKET / "engraving.png"), width=1500))
print("Review the crop with the full page if a marking extends beyond it.")

Review the crop with the full page if a marking extends beyond it.


In [4]:
for filename in ("score.svg", "score-enriched.svg"):
    svg_root = ET.parse(PACKET / filename).getroot()
    note = next(e for e in svg_root.iter() if e.get("id") == "d1e60")
    print(filename, dict(note.attrib))
    print("UTF-8 bytes:", (PACKET / filename).stat().st_size)
print("MEI context bytes:", (PACKET / "context.xml").stat().st_size)
print("context.xml is read-only context, not an editable/importable MEI document.")

score.svg {'id': 'd1e60', 'class': 'note'}
UTF-8 bytes: 49731
score-enriched.svg {'id': 'd1e60', 'class': 'note', 'data-pname': 'c', 'data-oct': '3', 'data-dur': '8', 'data-stem.dir': 'up'}
UTF-8 bytes: 52983
MEI context bytes: 10302
context.xml is read-only context, not an editable/importable MEI document.


## 15–20 — Manual edit and live refresh

Use the live-editing notebook to restore the opening dynamic and tempo
manually. Keep the same starting packet if comparing A/B: both conversations
must receive identical inputs. After changing the input MEI, use a **new**
packet folder and rerun the export cell before the model/candidate task.
An existing packet is preserved. The candidate step checks for stale input.

## 20–30 — Let the model inspect the slur phrases

Choose one condition. Use the same model and a fresh conversation for A/B.
A receives `source-system.png`, `engraving.png`, `context.xml`, `prompt-A.md`.
B gets those same images/context and `score.svg` as **source text**, using
`prompt-B.md`. Optional C substitutes `score-enriched.svg` and `prompt-C.md`.
Do not attach both SVG variants to B or give one condition the teacher key.
Record how the extension actually sends SVG: text, raster preview, both, unknown.

The question asks the model to infer two opening phrase divisions from the
print, identify existing MEI endpoints and propose a small patch. It does
not supply the teacher's endpoints. Compare the explanation and targets,
not merely whether a response sounds plausible. With text-only access, use
the verified forte prompt in the live-editing notebook as the fallback.

In [5]:
CONDITION = "B"  # A: images+MEI; B: add default SVG text; C: enriched SVG text.
print((PACKET / f"prompt-{CONDITION}.md").read_text())

Compare source-system.png (IMSLP 51556, printed p.8, first system)
with engraving.png (our current Schumann Op.68 No.10 encoding). Read context.xml,
a READ-ONLY fragment with scoreDef and MEI labels 1–5; it is not a complete MEI.
The pickup is label 1; labels 2 and 3 are the first two full bars.

Focus ONLY on the slur phrases beginning in that pickup and those two full bars.
Identify up to two missing or differently attached printed slurs. For each:
state what you see in the source, the staff, start/end notes and existing MEI
IDs, the corresponding current-engraving discrepancy, and any uncertainty.
If you cannot inspect an attachment or distinguish an endpoint, say so.
Do not infer the source reading from a familiar edition or the current metadata.
Use only the supplied task files; do not consult answer keys, other editions or the web.

Then propose the smallest MEI 5.1 insertion/correction and an explanation for
editorial review. Preserve all existing pitches, rhythms, voices, IDs, 

## Optional tool-assisted candidate loop

After discussing its observations, you may let the model edit a **candidate**
copy and run local rendering/checks, while preserving `work.mei`. The cell
below creates that adjacent copy only when explicitly enabled; it never
overwrites an existing one. Its relative image paths still resolve.
Tell the agent the exact candidate and output paths. Give it the packet and
the task, restrict the patch to the identified marks, then inspect the diff.
A model may render via `export_svg_editorial_packet(candidate, NEW_PACKET)`;
use a new output directory for each attempt. Do not regenerate MEI from SVG.
In a repository checkout, `tools/export_svg_editorial.py --mei ... --output ...`
exposes the same function as a CLI; see `examples/model-prompts/svg-candidate-loop.md`.

In [6]:
CREATE_CANDIDATE = False
CANDIDATE = INPUT_MEI.with_name("candidate-svg.mei")
if CREATE_CANDIDATE:
    if svg_file_hash(INPUT_MEI) != MANIFEST["identity"]["input_mei_sha256"]:
        raise ValueError("Working MEI changed since export; make a fresh packet before creating the candidate")
    if CANDIDATE.exists():
        print("Existing candidate preserved:", CANDIDATE)
    else:
        with CANDIDATE.open("xb") as destination:
            destination.write(INPUT_MEI.read_bytes())
        print("Candidate created:", CANDIDATE)
else:
    print("No editable candidate created. Enable CREATE_CANDIDATE only for the candidate exercise.")

No editable candidate created. Enable CREATE_CANDIDATE only for the candidate exercise.


## 30–40 — Render, validate and make an editorial decision

Inspect the candidate against the print. Use the live-editing notebook's
`audit_editing_changes(INPUT_MEI, CANDIDATE)` guard with the pre-patch file,
and run CAMAT validation on the **candidate**, including Verovio:

```sh
camat-validate-mei /absolute/path/to/edition/mei/candidate-svg.mei --output-dir /absolute/path/to/edition/validation-svg --verovio
```

Re-render the candidate into a new packet and inspect both the picture and
encoded endpoints. Explain the accepted reading in an anchored annotation.
Copy `record-template.json` to a new file per condition and record the model,
actual input transport, response, patch, checks and human decision. No model
result or source approval is created by running this notebook alone.

In [7]:
print("Blank record:", PACKET / "record-template.json")
print("Provider calls made by this notebook:", MANIFEST["provider_calls"])

Blank record: /tmp/camat-schumann-svg/notebook/schumann_edition_work/edition/experiments/svg-01/record-template.json
Provider calls made by this notebook: 0
